# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{getattr(metadata, 'name')}: {getattr(metadata, 'description')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The following code snippet lists all record sets and their available fields by their `@id`.

In [ ]:
# List all record sets by @id and their associated fields
record_sets = []
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    record_sets = metadata.recordSet
elif hasattr(metadata, 'record_sets') and metadata.record_sets:
    record_sets = metadata.record_sets

# Print available record set @ids and each field/column @id
if not record_sets:
    print("No record sets found in the metadata.")
else:
    for i, rs in enumerate(record_sets):
        rs_id = getattr(rs, '@id', f'_recordset_{i}')
        print(f"RecordSet {i+1} @id: {rs_id}")
        if hasattr(rs, 'field'):
            fields = rs.field
            print("  Fields:")
            for field in fields:
                field_id = getattr(field, '@id', None)
                print(f"   - {field_id}")
        if hasattr(rs, 'column'):
            columns = rs.column
            print("  Columns:")
            for col in columns:
                col_id = getattr(col, '@id', None)
                print(f"   - {col_id}")
        print()
# If the recordsets list is empty, try to enumerate ids from the datafiles
if not record_sets:
    print("Attempting to infer record sets from available distributions (data files):")
    if hasattr(metadata, 'distribution'):
        for dist in metadata.distribution:
            dist_id = getattr(dist, '@id', None)
            print(f"- Distribution @id: {dist_id}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use record set and field `@id`s from the overview.

Since some datasets provide records via datafiles but do not list explicit `recordSet` entities, we will attempt to enumerate available record sets by inspecting the dataset. 

Below we load records for each detected record set (using its `@id`).

In [ ]:
# Because recordSet may be empty in this dataset, we attempt to get all available record sets
available_record_sets = []
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    available_record_sets = [getattr(rs, '@id', None) for rs in metadata.recordSet]
elif hasattr(metadata, 'record_sets') and metadata.record_sets:
    available_record_sets = [getattr(rs, '@id', None) for rs in metadata.record_sets]
        
# If none, check for fallback available via dataset.records()
if not available_record_sets:
    # mlcroissant will expose supported recordSet ids via dataset.record_sets property
    if hasattr(dataset, 'record_sets'):
        available_record_sets = list(dataset.record_sets)
    else:
        print("No record sets found via Croissant metadata.")
        available_record_sets = []

print(f"Available record sets: {available_record_sets}")

dataframes = {}
for record_set_id in available_record_sets:
    try:
        print(f"Loading records for record set @id: {record_set_id}")
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Columns: {df.columns.tolist()}")
            display(df.head())
        else:
            print(f"No records found for record set: {record_set_id}")
    except Exception as e:
        print(f"Failed to load records for {record_set_id}: {e}")

# Choose one record set for further EDA
if dataframes:
    selected_record_set_id = list(dataframes.keys())[0]
    print(f"\nSelected record set for further analysis: {selected_record_set_id}")
    print(f"Available columns: {dataframes[selected_record_set_id].columns.tolist()}")
else:
    selected_record_set_id = None
    print("No dataframes available for further analysis.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, and grouping data by key attributes.

In [ ]:
# Select a numeric field by its `@id` for further analysis
import numpy as np
if selected_record_set_id is not None:
    df = dataframes[selected_record_set_id]
    
    # Try to auto-detect numeric columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_cols:
        # Try to force conversion for likely numeric columns
        potential_numeric = [col for col in df.columns if any(word in col.lower() for word in ["count", "number", "abundance", "intensity", "mw", "weight", "%", "coverage", "peptide", "score", "pval", "pvalue", "modification"])]
        for col in potential_numeric:
            try:
                df[col] = pd.to_numeric(df[col], errors="coerce")
            except Exception:
                pass
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    print(f"Detected numeric columns: {numeric_cols}")

    if numeric_cols:
        numeric_field = numeric_cols[0]
        threshold = df[numeric_field].mean() if pd.notnull(df[numeric_field].mean()) else 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold} (mean):")
        display(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to select a categorical/group field, such as 'description', 'accession', or similar
        group_candidates = [col for col in df.columns if any(word in col.lower() for word in ["group", "category", "type", "class", "source", "sample", "description"])]
        group_field = group_candidates[0] if group_candidates else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            display(grouped_df.head())
    else:
        print("No numeric fields detected for EDA.")
else:
    print("No selected record set to analyze.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below we plot the distribution of the selected numeric field and, where possible, visualize normalized values.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set_id is not None and numeric_cols:
    plt.figure(figsize=(10,5))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    if f"{numeric_field}_normalized" in filtered_df:
        plt.figure(figsize=(10,5))
        sns.histplot(filtered_df[f"{numeric_field}_normalized"].dropna(), kde=True)
        plt.title(f"Distribution of Normalized {numeric_field} (filtered)")
        plt.xlabel(f"{numeric_field}_normalized")
        plt.ylabel("Count")
        plt.show()

    if group_field:
        plt.figure(figsize=(12,6))
        # For better results, plot means if group_field has many unique values
        group_means = df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False).head(20)
        sns.barplot(x=group_means.values, y=group_means.index)
        plt.title(f"Top 20 groups by mean {numeric_field}")
        plt.xlabel(f"Mean {numeric_field}")
        plt.ylabel(group_field)
        plt.tight_layout()
        plt.show()
else:
    print("No numeric data available for visualization.")

## 6. Conclusion
This notebook demonstrated how to load and explore a Croissant-described dataset using `mlcroissant`. We:
- Loaded the FAIR^2 schema and inspected dataset metadata
- Enumerated available record sets and columns by their `@id`
- Loaded records into pandas DataFrames using only `@id` references
- Performed basic exploratory data analysis (filtering, normalization, grouping)
- Visualized key distributions and group summaries

This workflow can be extended to more advanced analyses, ML modeling, or used as a template for exploring any Croissant-compatible dataset.
